In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
import enum
import pandas as pd

2023-11-02 10:05:28.543026: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-11-02 10:05:28.580821: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-11-02 10:05:28.581480: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-02 10:05:29.362138: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
num_points = 21
values_per_point = 3

model = Sequential()

model.add(Dense(512, activation='relu', input_shape=(num_points * values_per_point,), kernel_regularizer=l2(0.01)))
model.add(Dropout(0.3))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(25, activation='sigmoid'))

2023-11-02 10:05:31.702396: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-11-02 10:05:31.703097: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1956] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
data = pd.read_csv("kaggle_dataset_xyz_rotated.csv")
data.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,P,R,S,T,U,V,W,X,Y,Z
0,0.336513,0.573219,0.737598,0.722364,0.623556,0.524646,0.468336,0.451130,0.470516,0.336513,...,0,0,0,0,0,0,0,0,0,0
1,0.336357,0.525727,0.604424,0.627149,0.670503,0.497172,0.497921,0.525606,0.553749,0.336357,...,0,0,0,0,0,0,0,0,0,0
2,0.336166,0.569811,0.748705,0.707248,0.604747,0.513676,0.447124,0.425074,0.450105,0.336166,...,0,0,0,0,0,0,0,0,0,0
3,0.327314,0.570673,0.695457,0.678917,0.613407,0.498035,0.490187,0.510991,0.540480,0.327314,...,0,0,0,0,0,0,0,0,0,0
4,0.365060,0.651679,0.845355,0.821910,0.715768,0.554929,0.555831,0.597786,0.628526,0.365060,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Podziel dane na funkcje wejściowe X i etykiety y
X = data.iloc[:, :63]  # wszystkie kolumny oprócz ostatnich
y = data.iloc[:, 63:]  # tylko ostatnia kolumna

# Podziel dane na zbiory treningowe i testowe
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


# Skonwertuj DataFrame do numpy arrays
X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values
X_val = X_val.values
y_val = y_val.values

In [6]:

metrics = tf.keras.metrics.AUC(
    num_thresholds=200,
    curve='ROC',
    summation_method='interpolation',
    name=None,
    dtype=None,
    thresholds=None,
    multi_label=False,
    num_labels=None,
    label_weights=None,
    from_logits=False
)

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=[metrics])

In [7]:
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=30, batch_size=32)

Epoch 1/30
484/484 [==============================] - 3s 3ms/step - loss: 0.2523 - auc: 0.7841 - val_loss: 0.0845 - val_auc: 0.9738
Epoch 2/30
484/484 [==============================] - 1s 3ms/step - loss: 0.0924 - auc: 0.9593 - val_loss: 0.0589 - val_auc: 0.9831
Epoch 3/30
484/484 [==============================] - 1s 3ms/step - loss: 0.0699 - auc: 0.9783 - val_loss: 0.0498 - val_auc: 0.9906
Epoch 4/30
484/484 [==============================] - 1s 3ms/step - loss: 0.0616 - auc: 0.9827 - val_loss: 0.0420 - val_auc: 0.9932
Epoch 5/30
484/484 [==============================] - 1s 3ms/step - loss: 0.0571 - auc: 0.9845 - val_loss: 0.0429 - val_auc: 0.9914
Epoch 6/30
484/484 [==============================] - 1s 3ms/step - loss: 0.0548 - auc: 0.9857 - val_loss: 0.0393 - val_auc: 0.9943
Epoch 7/30
484/484 [==============================] - 1s 3ms/step - loss: 0.0521 - auc: 0.9863 - val_loss: 0.0365 - val_auc: 0.9949
Epoch 8/30
484/484 [==============================] - 1s 3ms/step - loss: 0.

In [8]:
loss = model.evaluate(X_test, y_test)

104/104 [==============================] - 0s 1ms/step - loss: 0.0346 - auc: 0.9946


In [58]:
model.save('model_rotated.h5')